# 06 Comparative analysis (A vs B vs C)

**Question.** How do B and C differ from A on the validation partition, with uncertainty?

**Type.** Formal comparison on validation predictions via paired bootstrap (resampling unit: image; optimistic because patient grouping is unknown). Set `RUN_IDS` explicitly: the notebook refuses to guess when an experiment has several completed runs.

**Caveat.** The validation partition was also used to select each model's checkpoint, so these numbers are selection-biased and a single training seed is not enough to separate real differences from seed noise. See [the experiment protocol](../docs/experiment-protocol.md).

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ct_iqa.evaluation.bootstrap import paired_bootstrap_difference
from ct_iqa.experiment import report
from ct_iqa.experiment.runs import read_predictions, select_run

IDS = {"A": "A_reference_resnet50", "B": "B_ranking_aware_resnet50", "C": "C_uncertainty_aware_resnet50"}
RUN_IDS = {"A": None, "B": None, "C": None}      # set to a run directory name when an experiment has several completed runs
runs = {k: select_run(IDS[k], RUN_IDS[k]) for k in IDS}
display(pd.DataFrame({"run": {k: (r.name if r else "PENDING (no completed run)") for k, r in runs.items()}}))
preds = {k: read_predictions(r) for k, r in runs.items() if r}


## Validation metrics per model (selected epoch, raw 0-4 scale)

In [ ]:
table = {k: json.loads((r / "metrics_validation.json").read_text()) for k, r in runs.items() if r}
if table:
    display(pd.DataFrame({k: {m: v[m] for m in report.POINT_METRICS} for k, v in table.items()}).round(4))


## Paired bootstrap differences (model minus A, 95% interval)

In [ ]:
def compare(other):
    if "A" not in preds or other not in preds:
        print(f"{other} vs A: PENDING (missing completed run)")
        return
    a, b = preds["A"], preds[other]
    assert a["filename"] == b["filename"], "runs must share the validation partition"
    res = paired_bootstrap_difference(a["label_0_4"], a["prediction_0_4"], b["prediction_0_4"], n_boot=2000, seed=0)
    display(pd.DataFrame(res).T[["difference_b_minus_a", "ci_low", "ci_high", "fraction_positive"]].round(4))
    report.plot_bootstrap_differences(res, f"{other} - A")
    plt.show()


compare("B")
compare("C")
